In [1]:
library(tidyverse)
library(readxl)
library(strpip)
rm(list = ls())

Warning message:
"package 'ggplot2' was built under R version 4.4.1"
Warning message:
"package 'purrr' was built under R version 4.4.1"
Warning message:
"package 'lubridate' was built under R version 4.4.1"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.5.1
v ggplot2   3.5.2     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.0.4     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'readxl' was built under R version 4.4.1"
Warning message:
"package 'rlang' was built under R version 4.4.1"

Attaching package: 'rlang'


The following objects are masked from 'package:purrr':

    %@%, flatten, flatten_chr, flatten_dbl, flat

## CANCER signatures

### MHG

In [4]:
file <- "published-signatures-human/resources/Sha_et_al_2018_JCO.xlsx"
sheets <- excel_sheets(file)
sheets

[1] "DEG_BLvsGCB"          "DEG_BLvsMHG"          "DEG_MHGvsGCB"        
[4] "DEG_MHGmycrVSGCBmycr" "DEG_MHGmycnVSGCB"

In [5]:
mhg_list <- list()
for(i in seq_along(sheets[c(2:5)])) {
    mhg_list[[i]] <- read_excel(file, sheet = sheets[i])
    if(i == 1) {
        mhg_list[[i]] <- mhg_list[[i]] %>% 
            mutate(logFC = -logFC) %>%
            arrange(desc(logFC))}}
names(mhg_list) <- c("MHG_vs_BL", "MHG_vs_GCB", "MYCr_MHG_vs_GCB", "MYCn_MHG_vs_GCB")

names(mhg_list)
mhg_list[[1]] %>% head(.)

[1] "MHG_vs_BL"       "MHG_vs_GCB"      "MYCr_MHG_vs_GCB" "MYCn_MHG_vs_GCB"

GeneSymbol,logFC,AveExpr,t,P.Value,adj.P.Val,B
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
EBI3,2.923349,13.12670,-18.64035,2.752899e-60,3.163081e-57,126.42057
TMEM119,2.801518,12.77097,-15.80724,1.457112e-46,3.842476e-44,95.07417
SERPINA9,2.792670,12.35720,-12.33486,5.523926e-31,2.284264e-29,59.52308
SLAMF1,2.687819,11.49098,-13.94062,5.728328e-38,5.295741e-36,75.45368
MAP3K8,2.609893,12.61605,-15.93702,3.551194e-47,1.038627e-44,96.47430
CDH11,2.504779,12.35438,-12.32852,5.873617e-31,2.422641e-29,59.46231


In [6]:
for(i in seq_along(mhg_list)) {
    total <- mhg_list[[i]] %>%
        filter(adj.P.Val < 0.05) %>%
        .$`GeneSymbol`
    up <- mhg_list[[i]] %>%
        filter(adj.P.Val < 0.05, logFC > 0) %>%
        .$`GeneSymbol`
    down <- mhg_list[[i]] %>%
        filter(adj.P.Val < 0.05, logFC < 0) %>%
        .$`GeneSymbol`
    assign(names(mhg_list)[i], total)
    assign(paste0(names(mhg_list)[i], "_UP"), up)
    assign(paste0(names(mhg_list)[i], "_DOWN"), down)
}

In [13]:
ls()

[1] "DHIT"                 "DHIT_DOWN"            "DHIT_UP"             
 [4] "LYMPH2CX_DLBCL_ABC"   "LYMPH2CX_DLBCL_GCB"   "MHG_vs_BL"           
 [7] "MHG_vs_BL_DOWN"       "MHG_vs_BL_UP"         "MHG_vs_GCB"          
[10] "MHG_vs_GCB_DOWN"      "MHG_vs_GCB_UP"        "MYCn_MHG_vs_GCB"     
[13] "MYCn_MHG_vs_GCB_DOWN" "MYCn_MHG_vs_GCB_UP"   "MYCr_MHG_vs_GCB"     
[16] "MYCr_MHG_vs_GCB_DOWN" "MYCr_MHG_vs_GCB_UP"   "REDDY_DLBCL_ABC"     
[19] "REDDY_DLBCL_GCB"      "down"                 "file"                
[22] "i"                    "mhg_list"             "sheets"              
[25] "sigdb"                "siglist"              "total"               
[28] "up"

In [14]:
mhg <- mget(ls(pattern = "MHG"))
mhg_db <- convert_list_to_df(mhg)
write_gmt(mhg_db, save_dir = "published-signatures-human/gmt/", save_name = "SHA_2018_MHG.gmt")


### DHIT

In [15]:
## Double hit signature in Burkitt's lymphoma with MYC and BCL2 rearrangements
DHIT_UP <- c(
  "OR13A1", "FAM216A", "MYC", "SLC25A27", "ALOX5", "UQCRH", "SUGCT", "SNHG7", "TNFSF8", "LINC00957", "PEG10", 
  "PIK3CD-AS2", "GAMT", "RPL6", "EIF4EBP3", "SNHG19", "QRSL1", "FHIT", "SLC29A2", "TERT", "SMARCB1", "RGCC", 
  "SNHG17", "JCHAIN", "SPTBN2", "ATF4", "CD24", "RPL35", "HAGHL", "CTD-3074O7.5", "WNK2", "AFMID", "CCDC78", 
  "RPL13", "RPL7", "SFXN4", "SGCE", "SMIM14", "LRRC75A-AS1", "HRK", "DANCR", "SYBU", "RPS8", "SNHG11", "NMRAL1", 
  "PPP1R14B", "MACROD1", "SOX9")
  
DHIT_DOWN <- c("MYEOV", "IL10RA", "GPR137B", "TLE4", "PARP15", "CCL17", "HMSD", "DOCK10", "MVP", 
  "ASS1P1", "GNG2", "CDK5R1", "ETV5", "RASGRF1", "ACPP", "COBLL1", "LY75", "ARPC2", "CFLAR", "AC104699.1", "GALNT6", 
  "VASP", "ARHGAP25", "SIGLEC14", "PTPRJ", "CR2", "CAB39", "HIVEP1", "RFFL", "ADTRP", "MIR155HG", "POU3F1", "VOPP1", 
  "BATF", "MREG", "STAT3", "TACC1", "IRF4", "ST8SIA4", "WDFY1", "ARID3B", "CCL22", "SIAH2", "SGPP2", "CPEB4", "CD80", 
  "SEMA7A", "ANKRD33B", "NCOA1", "BCL2A1", "DGKG", "ALS2", "LTA", "FCRL5", "EBI3", "IL21R")

DHIT <- c(DHIT_UP, DHIT_DOWN)

dhit <- mget(ls(pattern = "DHIT"))
dhit_db <- convert_list_to_df(dhit)
write_gmt(dhit_db, save_dir = "published-signatures-human/gmt/", save_name = "ENNISHI_2018_DHIT.gmt")


### LYMPH2CX DLBCL ABC GCB

In [17]:
`LYMPH2CX_DLBCL_ABC` <- c("MME", "SERPINA9", "ASB13", "MAML3", "ITPKB", "MYBL1", "S1PR2")
`LYMPH2CX_DLBCL_GCB` <- c("TNFRSF13B", "LIMD1", "IRF4", "CREB3L2", "PIM2", "CYB5R2", "RAB7L1", "CCDC50")

dlbcl_db <- convert_list_to_df(list(LYMPH2CX_DLBCL_ABC, LYMPH2CX_DLBCL_GCB))
write_gmt(dlbcl_db, save_dir = "published-signatures-human/gmt/", save_name = "LYMPH2CX_DLBCL_SUBGROUPS.gmt")


### REDDY DLBCL ABC GCB

In [18]:
`REDDY_DLBCL_ABC` <- c("BMF", "SH3BP5", "BLNK", "IL16", "IRF4", "PIM1", "CCND2", "ENTPD1", "FUT8", "PTPN1", "ETV6")
`REDDY_DLBCL_GCB` <- c("LMO2", "NEK6", "DENND3", "MME", "BCL6", "LRMP", "MYBL1", "ITPKB")

reddy_db <- convert_list_to_df(list(REDDY_DLBCL_ABC, REDDY_DLBCL_GCB))
write_gmt(reddy_db, save_dir = "published-signatures-human/gmt/", save_name = "REDDY_2017_DLBCL_SUBGROUPS.gmt")